# Enrichment and Validation

*Attach useful context and verify keys, coverage, ranges, and relationships between fields.*


## Enrichment Adds Context

Enrichment adds information that the received records did not contain, such as
a subject's department or a derived score proportion. The new fields must use
compatible definitions and units. A lookup with repeated keys can multiply
records; an incomplete lookup can introduce missing information.

Validation asks whether a stated rule holds after a transformation. Passing a
rule does not prove that every source measurement is accurate. These concepts
reconstruct the enrichment and validation explanations in `4_1.pptx`, slides 11–13.


## Check Lookup Cardinality and Coverage

`indicator=True` adds a merge status. `validate="many_to_one"` verifies the
lookup's key uniqueness, while the status reveals records with no lookup match.
Both checks are needed: a unique lookup can still be incomplete.


In [1]:
import pandas as pd

scores_df = pd.DataFrame(
    {
        "name": ["Jiyoung", "Hojun", "Hana"],
        "subject": ["Mathematics", "English", "Science"],
        "score": [98, 90, 85],
        "maximum_score": [100, 100, 100],
    }
)
lookup_df = pd.DataFrame(
    {
        "subject": ["Mathematics", "English"],
        "department": ["STEM", "Languages"],
    }
)
enriched_df = scores_df.merge(
    lookup_df, on="subject", how="left", validate="many_to_one", indicator=True
)
print(enriched_df.to_string(index=False))
print(
    "Unmatched subjects:",
    enriched_df.loc[enriched_df["_merge"] == "left_only", "subject"].tolist(),
)


   name     subject  score  maximum_score department    _merge
Jiyoung Mathematics     98            100       STEM      both
  Hojun     English     90            100  Languages      both
   Hana     Science     85            100        NaN left_only
Unmatched subjects: ['Science']


All three scores remain, but Science has no department. The left merge
makes that missing context visible instead of dropping Hana's observation.


## Validate Values and Cross-Field Relationships

Some rules concern one field, such as a nonnegative score. Others compare
fields: a score must not exceed its own `maximum_score`. `.all()` reduces a
row-wise Boolean condition to a single reported result.


In [2]:
checks = pd.Series(
    {
        "numeric_scores": pd.api.types.is_numeric_dtype(enriched_df["score"]),
        "unique_student_subject": not enriched_df.duplicated(["name", "subject"]).any(),
        "positive_maximum": enriched_df["maximum_score"].gt(0).all(),
        "score_within_own_maximum": (
            enriched_df["score"].ge(0)
            & enriched_df["score"].le(enriched_df["maximum_score"])
        ).all(),
        "complete_lookup": enriched_df["_merge"].eq("both").all(),
        "row_count_preserved": len(enriched_df) == len(scores_df),
    }
)
print("Validation results:")
print(checks)


Validation results:
numeric_scores               True
unique_student_subject       True
positive_maximum             True
score_within_own_maximum     True
complete_lookup             False
row_count_preserved          True
dtype: bool


The type, key, range, and row-count rules pass. Complete lookup coverage
is `False`, identifying the one remaining context issue.


## Revalidate After a Confirmed Lookup Update

The prepared case confirms Science's department as STEM. Rebuild the merge
with the updated lookup. Only after verifying positive maximum scores does
division produce the score proportion.


In [3]:
complete_lookup_df = pd.concat(
    [lookup_df, pd.DataFrame({"subject": ["Science"], "department": ["STEM"]})],
    ignore_index=True,
)
final_df = scores_df.merge(
    complete_lookup_df, on="subject", how="left", validate="many_to_one"
)
final_df["score_proportion"] = final_df["score"] / final_df["maximum_score"]
print(final_df.to_string(index=False))
print("Complete department coverage:", final_df["department"].notna().all())
print(
    "Proportions between zero and one:",
    final_df["score_proportion"].between(0, 1).all(),
)


   name     subject  score  maximum_score department  score_proportion
Jiyoung Mathematics     98            100       STEM              0.98
  Hojun     English     90            100  Languages              0.90
   Hana     Science     85            100       STEM              0.85
Complete department coverage: True
Proportions between zero and one: True


Coverage is now complete. The three proportions, 0.98, 0.90, and 0.85,
retain their relationship to the original grades and their 100-point maxima.


Validate the added fields as well as the original records. Check key cardinality, match coverage, row counts, and relationships between values before sharing the result.
